<p align="center">
<img src="Images/sorbonne_logo.png" alt="Logo" width="300"/>
</p>

# **Module 2 - Statistics & Data Transformation**

* **Author**: Elia Landini
* **Student ID**: 12310239
* **Course**: EESM2-Financial Economics 
* **Supervisor**: Jean-Bernard Chatelain
* **Reference Repository**: https://github.com/EliaLand/IRL-FAVAR_money_endogeneity

### **1) REQUIREMENTS SET-UP**

In [18]:
# Requirements.txt file installation
# !pip install -r requirements.txt

In [19]:
# Libraries import
import warnings
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
import seaborn as sns
import scipy.stats as stats
from scipy.stats import norm
from scipy.stats import levene
from scipy.stats import ks_2samp
from scipy.stats import kstest
from scipy.stats import pearsonr
from scipy.stats import gaussian_kde
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.filters.hp_filter import hpfilter
import sklearn.tree
import sklearn.metrics
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing 
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             accuracy_score, precision_recall_curve, auc, 
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.linear_model import (LinearRegression, LogisticRegression)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
import plotly.express as px
import openpyxl as pxl
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML
from IPython.display import Image
import itertools
from arch.unitroot import PhillipsPerron

### **2) DESCRIPTIVE STATISTICS (RAW)**

In [20]:
# df import
jp_aggregated_df = pd.read_csv("Data/Aggregated/jp_aggregated_df.csv")
jp_aggregated_df

,Country,Time,Monetary Aggregates - M1 (JPY),Monetary Aggregates - M2 (JPY),Monetary Aggregates - M3 (JPY),Total Credit - Private Non-Financial (%GDP),Total Credit - General Government (%GDP),Total Credit - Households & NPISHs (%GDP),Total Treasury Reserves (- Gold),10-Year Gov Bond Yields (%),...,Real GDP (billions chained 2015 JPY),Central Government Debt (% GDP),Domestic Private Debt Securities (% GDP),Domestic Public Debt Securities (% GDP),BoJ’s Total Assets (100 Million Yen),Loan Interest Rate (%),Deposit Interest Rate (%),1615.T-Price,10-Year US T-Bills Yield (%),CBOE-VIX
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,5.980000e+02,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900,JP,2025-12,NaN,NaN,NaN,NaN,NaN,NaN,1.252603e+06,2.06,...,591856.7,NaN,NaN,NaN,6777762.0,1.404,0.225,528.200012,4.143182,15.548182
901,JP,2026-01,NaN,NaN,NaN,NaN,NaN,NaN,1.259248e+06,2.24,...,NaN,NaN,NaN,NaN,6828680.0,NaN,0.227,598.299988,4.213500,16.179048
902,JP,2026-02,NaN,NaN,NaN,NaN,NaN,NaN,1.268656e+06,2.11,...,NaN,NaN,NaN,NaN,6837705.0,NaN,0.296,647.900024,4.125789,19.207000
903,JP,2026-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,6621318.0,NaN,NaN,NaN,4.245909,25.596818


In [21]:
# Metrics clusters for plotting
metrics_group = {
    "Monetary Aggregates" : ["Monetary Aggregates - M1 (JPY)", "Monetary Aggregates - M2 (JPY)", "Monetary Aggregates - M3 (JPY)"],
    "Credit Metrics" : ["Total Credit - General Government (%GDP)", "Total Credit - Households & NPISHs (%GDP)", "Total Credit - Private Non-Financial (%GDP)"],
    "Reserves" : ["Total Treasury Reserves (- Gold)"],
    "Monetary Policy Proxies (Yields)" : ["10-Year Gov Bond Yields (%)", "Call Money/Interbank Immediate (%)", "Est. 1-year Neutral Interest Rate (%)", "Est. 10-year Neutral Interest Rate (%)"], 
    "Exchange Rate" : ["USD-JPY reer CPI-based (Index 2015=100)", "JPY-USD Spot Exchange Rate"],
    "Output-Trends": ["Real GDP (billions chained 2015 JPY)"],
    "Consumption Prices": ["HICP (NSA)"],
    "Debt Metrics" : ["Central Government Debt (% GDP)", "Domestic Private Debt Securities (% GDP)", "Domestic Public Debt Securities (% GDP)"],
    "Banking Sector" : ["Loan Interest Rate (%)", "Deposit Interest Rate (%)", "1615.T-Price"],
    "Controls" : ["10-Year US T-Bills Yield (%)", "CBOE-VIX"],
    "BoJ Total Assets" : ["BoJ’s Total Assets (100 Million Yen)"]
}

In [22]:
# Descriptive statistics summary table - aggregate data
df = jp_aggregated_df.copy()
df.describe()

,Monetary Aggregates - M1 (JPY),Monetary Aggregates - M2 (JPY),Monetary Aggregates - M3 (JPY),Total Credit - Private Non-Financial (%GDP),Total Credit - General Government (%GDP),Total Credit - Households & NPISHs (%GDP),Total Treasury Reserves (- Gold),10-Year Gov Bond Yields (%),Call Money/Interbank Immediate (%),Est. 1-year Neutral Interest Rate (%),...,Real GDP (billions chained 2015 JPY),Central Government Debt (% GDP),Domestic Private Debt Securities (% GDP),Domestic Public Debt Securities (% GDP),BoJ’s Total Assets (100 Million Yen),Loan Interest Rate (%),Deposit Interest Rate (%),1615.T-Price,10-Year US T-Bills Yield (%),CBOE-VIX
count,7.670000e+02,7.460000e+02,5.270000e+02,732.000000,336.000000,732.000000,8.370000e+02,446.000000,488.000000,372.000000,...,384.000000,385.000000,277.000000,277.000000,3.360000e+02,387.000000,489.000000,218.000000,772.000000,436.000000
mean,2.799125e+14,3.659118e+14,9.357442e+14,161.884836,172.047321,53.637705,4.150285e+05,1.811004,1.246152,0.353968,...,535407.262500,129.955192,61.365237,140.770820,3.248560e+06,1.366514,1.347229,157.056883,5.815163,19.476344
std,2.845290e+14,3.038362e+14,3.323003e+14,26.992669,39.522164,14.944383,5.113915e+05,1.823192,2.183451,0.580089,...,36655.682733,57.276103,8.777222,37.063681,2.520776e+06,0.687633,2.239623,99.039048,2.935594,7.392915
min,4.021870e+12,3.564500e+12,2.952000e+14,110.800000,88.300000,19.900000,5.980000e+02,-0.280000,-0.071000,-0.420000,...,461491.900000,38.176187,46.447950,63.163390,6.800640e+05,0.465000,0.003250,67.772179,0.623636,10.125455
25%,5.802214e+13,4.820300e+13,7.623000e+14,139.175000,142.200000,44.325000,1.332832e+04,0.568750,0.001000,-0.260500,...,503947.575000,80.478284,55.154430,111.204600,1.187333e+06,0.789000,0.029000,103.542906,3.889679,14.114361
50%,1.518072e+14,3.102086e+14,9.926000e+14,160.850000,183.250000,60.150000,8.176792e+04,1.333500,0.091000,0.243500,...,535754.100000,128.599668,59.193390,157.152800,1.509510e+06,1.309000,0.120000,126.242744,5.440227,17.685287
75%,4.826217e+14,6.484915e+14,1.131859e+15,179.075000,206.800000,64.950000,9.827994e+05,1.954000,0.766435,0.864250,...,568025.350000,192.112850,68.579800,176.609800,5.619536e+06,1.696000,0.956000,155.226971,7.527175,23.098703
max,1.081546e+15,9.632187e+14,1.597004e+15,213.600000,231.500000,70.200000,1.371116e+06,8.032000,8.278130,1.380000,...,593799.900000,216.135005,78.846620,217.026000,7.648115e+06,4.043000,8.347500,647.900024,15.323810,62.668947


### **3) NON-STATIONARITY CORRECTIONS**

In [23]:
# Non-Stationarity Corrections - Log-Difference Transformations
# (!!!) Basically we need a detrending transformation for all the variables as expected, given the undisputable presence of unit-root root and so non-sttaionarity
# (!!!) Autocorrelation is also evident, suggesting a marked time-dependent component, and so a trend, so we'll opt for both log transformations as well as first differences 
df = jp_aggregated_df.copy()
trans_df = df[["Country", "Time"]].copy()

# Transformations: 
# - Monetary Aggregates: I(1) nominal levels (levels non-stationary, but first-differences are I(0), stationary)
# - Reserves: I(1), level series of policy shocks 
# - Exchange Rate: Log-difference (returns)
# - Consumption Prices: Log-difference (inflation)
# - Banking metruics: Log-difference (returns)
# - BoJ’s Total Assets: CA smoothing 
log_transformed_variables = ["Monetary Aggregates - M1 (JPY)", "Monetary Aggregates - M2 (JPY)", "Monetary Aggregates - M3 (JPY)",
                             "Total Treasury Reserves (- Gold)",
                             "USD-JPY reer CPI-based (Index 2015=100)", "JPY-USD Spot Exchange Rate",
                             "HICP (SA)",
                             "1615.T-Price",
                             "BoJ’s Total Assets (100 Million Yen)", 
                             "Call Money/Interbank Immediate (%)"
                            ]

# Log-Difference Transformed Variables and Annualization (percentage change over 12 months)
for var in log_transformed_variables:
    if var != "HICP (SA)":
        trans_df[f"LogDiff-{var}"] = np.log(df[f"{var}"]).diff()
    else:
        trans_df[f"LogDiff-{var}"] = np.log(df[f"{var}"]).diff()*12*100
trans_df


d:\Conda\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
d:\Conda\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Country,Time,LogDiff-Monetary Aggregates - M1 (JPY),LogDiff-Monetary Aggregates - M2 (JPY),LogDiff-Monetary Aggregates - M3 (JPY),LogDiff-Total Treasury Reserves (- Gold),LogDiff-USD-JPY reer CPI-based (Index 2015=100),LogDiff-JPY-USD Spot Exchange Rate,LogDiff-HICP (SA),LogDiff-1615.T-Price,LogDiff-BoJ’s Total Assets (100 Million Yen),LogDiff-Call Money/Interbank Immediate (%)
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
900,JP,2025-12,NaN,NaN,NaN,0.005796,-0.012713,0.004976,NaN,0.043927,-0.029329,0.152955
901,JP,2026-01,NaN,NaN,NaN,0.005291,-0.012815,0.004706,NaN,0.124617,0.007484,0.267736
902,JP,2026-02,NaN,NaN,NaN,0.007443,0.000443,-0.009937,NaN,0.079644,0.001321,0.000000
903,JP,2026-03,NaN,NaN,NaN,NaN,NaN,0.022826,NaN,NaN,-0.032158,NaN


In [24]:
# Non-Stationarity Corrections - AR(1) detrending 
df = jp_aggregated_df.copy()
trans_df = trans_df.copy()

# Transformations: 
# - Credit Metrics: AR(1) detrending (cyclical credit gap), we want to remove the persistence of credit t-1, we can reduce this relatiosnship to:
# (!!!) c_t = α + ρc_t−1​ + ε_t, we take the residuals 
# (!!!) first differences are to aggresive for credit metrics, they destroy medium-term cycles, signal-to-noise
# (!!!) credit metrics tend to be highly-persistent I(0) and near-unit-root stationary (so it might result non-stationary, but it is actually slowly mean reverting, we want to keep cyclical deviations)
# - Monetary Policy Proxies: AR(1) detrending (policy shocks), rates are persistent but not truly I(1)
# - Debt Metrics: same reasoning as for credit 
# - Banking metrics: same reasoning as for yields
# - VIX index
ar1detrend_transformed_variables = ["Total Credit - General Government (%GDP)", "Total Credit - Households & NPISHs (%GDP)", "Total Credit - Private Non-Financial (%GDP)",
                                    "10-Year Gov Bond Yields (%)", "Call Money/Interbank Immediate (%)", "Est. 1-year Neutral Interest Rate (%)", "Est. 10-year Neutral Interest Rate (%)",
                                    "Central Government Debt (% GDP)", "Domestic Private Debt Securities (% GDP)", "Domestic Public Debt Securities (% GDP)",
                                    "Loan Interest Rate (%)", "Deposit Interest Rate (%)",
                                    "10-Year US T-Bills Yield (%)", "CBOE-VIX"
                                   ]

# AR(1) detrending Transformed Variables 
for var in ar1detrend_transformed_variables:
# Lag-1 Reg
# (!!!) It triggers an errors as we don't specificy the model time index and frequency. It should be harmless, it only deactivates forecasting features.
    model = AutoReg(df[f"{var}"].dropna(), lags=1, old_names=False).fit()
# Residuals 
    trans_df[f"AR(1)detrend-{var}"] = df[f"{var}"] - model.fittedvalues
trans_df

d:\Conda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Conda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Conda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Conda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use

,Country,Time,LogDiff-Monetary Aggregates - M1 (JPY),LogDiff-Monetary Aggregates - M2 (JPY),LogDiff-Monetary Aggregates - M3 (JPY),LogDiff-Total Treasury Reserves (- Gold),LogDiff-USD-JPY reer CPI-based (Index 2015=100),LogDiff-JPY-USD Spot Exchange Rate,LogDiff-HICP (SA),LogDiff-1615.T-Price,...,AR(1)detrend-Call Money/Interbank Immediate (%),AR(1)detrend-Est. 1-year Neutral Interest Rate (%),AR(1)detrend-Est. 10-year Neutral Interest Rate (%),AR(1)detrend-Central Government Debt (% GDP),AR(1)detrend-Domestic Private Debt Securities (% GDP),AR(1)detrend-Domestic Public Debt Securities (% GDP),AR(1)detrend-Loan Interest Rate (%),AR(1)detrend-Deposit Interest Rate (%),AR(1)detrend-10-Year US T-Bills Yield (%),AR(1)detrend-CBOE-VIX
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900,JP,2025-12,NaN,NaN,NaN,0.005796,-0.012713,0.004976,NaN,0.043927,...,0.084282,0.002655,0.002863,NaN,NaN,NaN,0.264511,0.004569,0.041292,-4.176314
901,JP,2026-01,NaN,NaN,NaN,0.005291,-0.012815,0.004706,NaN,0.124617,...,0.176940,NaN,NaN,NaN,NaN,NaN,NaN,0.006569,0.062536,0.032409
902,JP,2026-02,NaN,NaN,NaN,0.007443,0.000443,-0.009937,NaN,0.079644,...,0.007362,NaN,NaN,NaN,NaN,NaN,NaN,0.073583,-0.095179,2.525659
903,JP,2026-03,NaN,NaN,NaN,NaN,NaN,0.022826,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.112260,6.349080


In [25]:
# Non-Stationarity Corrections - HP-filter cycle 
df = jp_aggregated_df.copy()
trans_df = trans_df.copy()

# Transformations: 
# - Output Trends: HP-filter cycle, trend smoothing
hpfilter_transformed_variables = ["Real GDP (billions chained 2015 JPY)"
                                 ]

# HP-filter cycle Transformed Variables 
for var in hpfilter_transformed_variables:    
    cycle, trend = hpfilter(
        np.log(df[f"{var}"]).dropna(),
        lamb=1600
    )
    trans_df[f"HPfilter-{var}"] = cycle
trans_df

,Country,Time,LogDiff-Monetary Aggregates - M1 (JPY),LogDiff-Monetary Aggregates - M2 (JPY),LogDiff-Monetary Aggregates - M3 (JPY),LogDiff-Total Treasury Reserves (- Gold),LogDiff-USD-JPY reer CPI-based (Index 2015=100),LogDiff-JPY-USD Spot Exchange Rate,LogDiff-HICP (SA),LogDiff-1615.T-Price,...,AR(1)detrend-Est. 1-year Neutral Interest Rate (%),AR(1)detrend-Est. 10-year Neutral Interest Rate (%),AR(1)detrend-Central Government Debt (% GDP),AR(1)detrend-Domestic Private Debt Securities (% GDP),AR(1)detrend-Domestic Public Debt Securities (% GDP),AR(1)detrend-Loan Interest Rate (%),AR(1)detrend-Deposit Interest Rate (%),AR(1)detrend-10-Year US T-Bills Yield (%),AR(1)detrend-CBOE-VIX,HPfilter-Real GDP (billions chained 2015 JPY)
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900,JP,2025-12,NaN,NaN,NaN,0.005796,-0.012713,0.004976,NaN,0.043927,...,0.002655,0.002863,NaN,NaN,NaN,0.264511,0.004569,0.041292,-4.176314,-0.001618
901,JP,2026-01,NaN,NaN,NaN,0.005291,-0.012815,0.004706,NaN,0.124617,...,NaN,NaN,NaN,NaN,NaN,NaN,0.006569,0.062536,0.032409,NaN
902,JP,2026-02,NaN,NaN,NaN,0.007443,0.000443,-0.009937,NaN,0.079644,...,NaN,NaN,NaN,NaN,NaN,NaN,0.073583,-0.095179,2.525659,NaN
903,JP,2026-03,NaN,NaN,NaN,NaN,NaN,0.022826,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.112260,6.349080,NaN


In [26]:
# Transformed df to csv
jp_trans_df = trans_df.copy()
jp_trans_df.to_csv("Data/Transformed/jp_trans_df.csv", index=False)

### **4) DESCRIPTIVE STATISTICS (TRANSFORMED)**

In [27]:
# Descriptive statistics summary table - Raw Data
df = jp_aggregated_df.copy()
df.describe()

,Monetary Aggregates - M1 (JPY),Monetary Aggregates - M2 (JPY),Monetary Aggregates - M3 (JPY),Total Credit - Private Non-Financial (%GDP),Total Credit - General Government (%GDP),Total Credit - Households & NPISHs (%GDP),Total Treasury Reserves (- Gold),10-Year Gov Bond Yields (%),Call Money/Interbank Immediate (%),Est. 1-year Neutral Interest Rate (%),...,Real GDP (billions chained 2015 JPY),Central Government Debt (% GDP),Domestic Private Debt Securities (% GDP),Domestic Public Debt Securities (% GDP),BoJ’s Total Assets (100 Million Yen),Loan Interest Rate (%),Deposit Interest Rate (%),1615.T-Price,10-Year US T-Bills Yield (%),CBOE-VIX
count,7.670000e+02,7.460000e+02,5.270000e+02,732.000000,336.000000,732.000000,8.370000e+02,446.000000,488.000000,372.000000,...,384.000000,385.000000,277.000000,277.000000,3.360000e+02,387.000000,489.000000,218.000000,772.000000,436.000000
mean,2.799125e+14,3.659118e+14,9.357442e+14,161.884836,172.047321,53.637705,4.150285e+05,1.811004,1.246152,0.353968,...,535407.262500,129.955192,61.365237,140.770820,3.248560e+06,1.366514,1.347229,157.056883,5.815163,19.476344
std,2.845290e+14,3.038362e+14,3.323003e+14,26.992669,39.522164,14.944383,5.113915e+05,1.823192,2.183451,0.580089,...,36655.682733,57.276103,8.777222,37.063681,2.520776e+06,0.687633,2.239623,99.039048,2.935594,7.392915
min,4.021870e+12,3.564500e+12,2.952000e+14,110.800000,88.300000,19.900000,5.980000e+02,-0.280000,-0.071000,-0.420000,...,461491.900000,38.176187,46.447950,63.163390,6.800640e+05,0.465000,0.003250,67.772179,0.623636,10.125455
25%,5.802214e+13,4.820300e+13,7.623000e+14,139.175000,142.200000,44.325000,1.332832e+04,0.568750,0.001000,-0.260500,...,503947.575000,80.478284,55.154430,111.204600,1.187333e+06,0.789000,0.029000,103.542906,3.889679,14.114361
50%,1.518072e+14,3.102086e+14,9.926000e+14,160.850000,183.250000,60.150000,8.176792e+04,1.333500,0.091000,0.243500,...,535754.100000,128.599668,59.193390,157.152800,1.509510e+06,1.309000,0.120000,126.242744,5.440227,17.685287
75%,4.826217e+14,6.484915e+14,1.131859e+15,179.075000,206.800000,64.950000,9.827994e+05,1.954000,0.766435,0.864250,...,568025.350000,192.112850,68.579800,176.609800,5.619536e+06,1.696000,0.956000,155.226971,7.527175,23.098703
max,1.081546e+15,9.632187e+14,1.597004e+15,213.600000,231.500000,70.200000,1.371116e+06,8.032000,8.278130,1.380000,...,593799.900000,216.135005,78.846620,217.026000,7.648115e+06,4.043000,8.347500,647.900024,15.323810,62.668947


In [28]:
# Descriptive statistics summary table - Transformed Data
df = jp_trans_df.copy()
df.describe()

d:\Conda\Lib\site-packages\numpy\core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,LogDiff-Monetary Aggregates - M1 (JPY),LogDiff-Monetary Aggregates - M2 (JPY),LogDiff-Monetary Aggregates - M3 (JPY),LogDiff-Total Treasury Reserves (- Gold),LogDiff-USD-JPY reer CPI-based (Index 2015=100),LogDiff-JPY-USD Spot Exchange Rate,LogDiff-HICP (SA),LogDiff-1615.T-Price,LogDiff-BoJ’s Total Assets (100 Million Yen),LogDiff-Call Money/Interbank Immediate (%),...,AR(1)detrend-Est. 1-year Neutral Interest Rate (%),AR(1)detrend-Est. 10-year Neutral Interest Rate (%),AR(1)detrend-Central Government Debt (% GDP),AR(1)detrend-Domestic Private Debt Securities (% GDP),AR(1)detrend-Domestic Public Debt Securities (% GDP),AR(1)detrend-Loan Interest Rate (%),AR(1)detrend-Deposit Interest Rate (%),AR(1)detrend-10-Year US T-Bills Yield (%),AR(1)detrend-CBOE-VIX,HPfilter-Real GDP (billions chained 2015 JPY)
count,766.000000,745.000000,526.000000,830.000000,673.000000,662.000000,663.000000,217.000000,335.000000,390.000000,...,3.710000e+02,3.710000e+02,3.840000e+02,2.760000e+02,2.760000e+02,3.860000e+02,4.880000e+02,7.710000e+02,4.350000e+02,3.840000e+02
mean,0.007285,0.007516,0.003210,0.008343,0.000112,-0.001229,2.370126,0.005592,0.006306,NaN,...,-8.259341e-17,5.163584e-16,-1.617040e-13,-1.722294e-14,-9.651539e-14,-1.015883e-15,4.085985e-16,2.718678e-15,1.263459e-14,-2.496383e-13
std,0.009906,0.006203,0.003104,0.037146,0.023527,0.025882,6.175389,0.073808,0.043007,NaN,...,1.895410e-02,2.475497e-02,2.255173e+00,1.835691e+00,4.016133e+00,1.067607e-01,1.434001e-01,2.750723e-01,3.927524e+00,9.385260e-03
min,-0.051049,-0.018761,-0.005262,-0.178330,-0.072705,-0.105227,-23.983093,-0.359858,-0.286708,-inf,...,-6.408364e-02,-8.371393e-02,-1.428292e+01,-1.286845e+01,-1.744624e+01,-3.052890e-01,-8.481754e-01,-1.714061e+00,-1.045059e+01,-6.170430e-02
25%,0.002267,0.002399,0.001218,-0.003341,-0.014098,-0.014824,-0.707249,-0.034887,-0.007289,-0.039443,...,2.389930e-03,2.392327e-03,-4.392336e-01,-2.213836e-01,-7.402428e-01,-6.571764e-02,-7.040450e-03,-1.385214e-01,-1.902298e+00,-4.201726e-03
50%,0.005488,0.006538,0.002277,0.005320,-0.002553,0.000390,1.129541,0.006349,0.008410,0.000000,...,3.679743e-03,4.083852e-03,-4.261780e-01,-1.230065e-01,-4.475790e-01,1.934661e-03,3.339265e-03,-7.109904e-03,-5.540305e-01,4.546983e-04
75%,0.011421,0.011827,0.004838,0.015606,0.012435,0.014794,4.199134,0.056571,0.023703,0.021774,...,5.578048e-03,6.790850e-03,-4.107848e-01,1.056187e-01,-3.203786e-01,7.012810e-02,1.266116e-02,1.472796e-01,1.029492e+00,4.829309e-03
max,0.086008,0.030447,0.019008,0.461758,0.110268,0.080657,50.654876,0.247112,0.280048,inf,...,7.952938e-02,1.067973e-01,1.798564e+01,1.265492e+01,3.242200e+01,2.882793e-01,1.248182e+00,1.634356e+00,3.813145e+01,2.462707e-02
